In [0]:
from pyspark.sql import functions as F

bronze_merchants = spark.table("fintech_fraud_risk.bronze.bronze_merchants")
bronze_devices   = spark.table("fintech_fraud_risk.bronze.bronze_devices")
bronze_login     = spark.table("fintech_fraud_risk.bronze.bronze_login_events")
bronze_fraud     = spark.table("fintech_fraud_risk.bronze.bronze_fraud_alerts")
bronze_cb        = spark.table("fintech_fraud_risk.bronze.bronze_chargebacks")

silver_customers = spark.table("fintech_fraud_risk.silver.customers")  # already written

all_txns = spark.table("fintech_fraud_risk.bronze.bronze_transactions").select("transaction_id") \
    .union(spark.table("fintech_fraud_risk.bronze.bronze_transaction_cdc").select("transaction_id")).distinct()

# merchants
silver_merchants = (
    bronze_merchants
    .withColumn("city", F.initcap(F.trim("city")))
    .dropDuplicates(["merchant_id"])
)
silver_merchants.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.silver.merchants")

# devices — FK against silver.customers
silver_devices = (
    bronze_devices
    .dropDuplicates(["device_id"])
    .join(silver_customers.select("customer_id"), "customer_id", "inner")
)
silver_devices.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.silver.devices")

# login_events — FK against silver.customers + silver.devices
silver_login = (
    bronze_login
    .withColumn("login_timestamp", F.to_timestamp("login_timestamp"))
    .dropDuplicates(["login_id"])
    .join(silver_customers.select("customer_id"), "customer_id", "inner")
    .join(silver_devices.select("device_id"), "device_id", "inner")
)
silver_login.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.silver.login_events")

# fraud_alerts — FK against unioned transactions
silver_fraud = (
    bronze_fraud
    .dropDuplicates(["alert_id"])
    .join(all_txns, "transaction_id", "inner")
)
silver_fraud.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.silver.fraud_alerts")

# chargebacks — type cast + FK against unioned transactions
silver_chargebacks = (
    bronze_cb
    .withColumn("chargeback_amount", F.col("chargeback_amount").cast("decimal(12,2)"))
    .dropDuplicates(["chargeback_id"])
    .join(all_txns, "transaction_id", "inner")
)
silver_chargebacks.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.silver.chargebacks")

print("Done — silver schema now has all tables.")

In [0]:
# analyize data after transferring

tables = [
    ("customers", "fintech_fraud_risk.bronze.bronze_customers", "fintech_fraud_risk.silver.customers"),
    ("merchants", "fintech_fraud_risk.bronze.bronze_merchants", "fintech_fraud_risk.silver.merchants"),
    ("devices", "fintech_fraud_risk.bronze.bronze_devices", "fintech_fraud_risk.silver.devices"),
    ("login_events", "fintech_fraud_risk.bronze.bronze_login_events", "fintech_fraud_risk.silver.login_events"),
    ("fraud_alerts", "fintech_fraud_risk.bronze.bronze_fraud_alerts", "fintech_fraud_risk.silver.fraud_alerts"),
    ("chargebacks", "fintech_fraud_risk.bronze.bronze_chargebacks", "fintech_fraud_risk.silver.chargebacks"),
    ("transactions","fintech_fraud_risk.bronze.bronze_transactions", "fintech_fraud_risk.silver.transactions")
]
for name, b_tbl, s_tbl in tables:
    b, s = spark.table(b_tbl).count(), spark.table(s_tbl).count()
    print(f"{name:15s} Bronze {b:>8,} -> Silver {s:>8,}  (pass rate {s/b*100:.2f}%)")

In [0]:
%sql
SHOW TABLES IN fintech_fraud_risk.silver;